In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

# training process
import algorithm.algo2 as algo2
reload(algo2)

# evaluation
import algorithm.kmeans as kmeans
reload(kmeans)
import eval_metrics.ac

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args = parser.parse_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

In [38]:
CLUSTER_NUM = 2
LAMBDA = 1

# CLUSTER_NUM = 2, LAMBDA = 2, [4,20], 100 iter -> 100%
# [ 8 11] 80 iter -> 87.5%

In [39]:
# read data
mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)
mat -= torch.mean(mat)

print(f'[Using classes] {np.unique(np.array(label))}')

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T
A.to(device)
pass

[Using classes] [10 12]


In [40]:
reload(algo2)
H = torch.randn(A.shape[0], CLUSTER_NUM)
H.to(device)

H_out = algo2.algo2(A=A, H=H, 
                    k=CLUSTER_NUM, lamda=LAMBDA, 
                    num_iter=100,
                    print_every=10)

iter 10 done, loss = 1449668.625, step = 0.0007995319319888949
iter 20 done, loss = 1443662.375, step = 0.0008263951167464256
iter 30 done, loss = 1442425.625, step = 0.0008285080548375845
iter 40 done, loss = 1440676.0, step = 0.0008181942976079881
iter 50 done, loss = 1436542.125, step = 0.0007849623798392713
iter 60 done, loss = 1433224.0, step = 0.0007423548377119005
iter 70 done, loss = 1432602.75, step = 0.0007220961851999164
iter 80 done, loss = 1432558.875, step = 0.0007166629075072706
iter 90 done, loss = 1432556.25, step = 0.0007154506165534258
iter 100 done, loss = 1432556.5, step = 0.0007151915924623609


In [42]:
reload(kmeans)
pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM)
acc = eval_metrics.ac.ac(label, pred_labels)
print(f'[Accuracy] {acc}')

[Accuracy] 1.0
